In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("RetailIntelligence-TP4-KPI")
    .master("local[*]").config("spark.hadoop.fs.defaultFS", "file:///")
    .getOrCreate()
)

DATA_ROOT = (
    Path.home()
    / "retail-intelligence"
    / "tp3"
    / "data"
)

silver_path = (
    DATA_ROOT
    / "silver-transformed"
    / "sales_by_store_date"
)

gold_path = (
    DATA_ROOT
    / "gold"
    / "sales_performance_by_store"
)

print("Spark :", spark.version)
print("Silver :", silver_path.exists())
print("Gold :", gold_path.exists())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/21 00:02:48 WARN Utils: Your hostname, NAVID-DELL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/21 00:02:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/navid/retail-intelligence/tp4/retail_intel_tp4_venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/21 00:02:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark : 4.3.0
Silver : True
Gold : True


In [2]:
# Chargement des jeux sélectionnés

df_silver = spark.read.parquet(str(silver_path))
df_gold = spark.read.parquet(str(gold_path))

print("Nombre de lignes Silver :", df_silver.count())
print("Nombre de lignes Gold :", df_gold.count())

print("\nSchéma Silver")
df_silver.printSchema()

print("\nSchéma Gold")
df_gold.printSchema()

Nombre de lignes Silver : 21430
Nombre de lignes Gold : 12

Schéma Silver
root
 |-- store_sk: long (nullable = true)
 |-- date_sk: long (nullable = true)
 |-- total_revenue_inc_tax: decimal(17,2) (nullable = true)
 |-- total_net_paid: decimal(17,2) (nullable = true)
 |-- total_net_profit: decimal(17,2) (nullable = true)
 |-- total_quantity_sold: long (nullable = true)
 |-- number_of_sales_lines: long (nullable = true)
 |-- number_of_tickets: long (nullable = true)
 |-- _grain: string (nullable = true)
 |-- _source_dataset: string (nullable = true)
 |-- _source_original_file: string (nullable = true)
 |-- _processing_stage: string (nullable = true)


Schéma Gold
root
 |-- store_sk: long (nullable = true)
 |-- revenue_rank: integer (nullable = true)
 |-- is_best_revenue_store: boolean (nullable = true)
 |-- total_revenue_inc_tax: decimal(17,2) (nullable = true)
 |-- global_revenue_inc_tax: decimal(27,2) (nullable = true)
 |-- revenue_share_pct: double (nullable = true)
 |-- total_net_pai

In [3]:
def check_columns(df, dataset_name, required_columns):

    print(f"\n===== {dataset_name} =====")

    schema = {
        field.name: field.dataType.simpleString()
        for field in df.schema.fields
    }

    for column in required_columns:

        if column in schema:
            print(f"[OK] {column} : {schema[column]}")
        else:
            print(f"[ABSENT] {column}")


check_columns(
    df_silver,
    "SILVER",
    [
        "store_sk",
        "date_sk",
        "total_revenue_inc_tax",
        "total_quantity_sold",
        "number_of_tickets"
    ]
)

check_columns(
    df_gold,
    "GOLD",
    [
        "store_sk",
        "total_revenue_inc_tax",
        "total_quantity_sold",
        "global_revenue_inc_tax",
        "revenue_rank",
        "revenue_share_pct"
    ]
)


===== SILVER =====
[OK] store_sk : bigint
[OK] date_sk : bigint
[OK] total_revenue_inc_tax : decimal(17,2)
[OK] total_quantity_sold : bigint
[OK] number_of_tickets : bigint

===== GOLD =====
[OK] store_sk : bigint
[OK] total_revenue_inc_tax : decimal(17,2)
[OK] total_quantity_sold : bigint
[OK] global_revenue_inc_tax : decimal(27,2)
[OK] revenue_rank : int
[OK] revenue_share_pct : double


In [4]:
# --------------------------------------------------
# KPI calculés à partir de Silver
# --------------------------------------------------

kpi_silver = df_silver.agg(
    F.sum("total_revenue_inc_tax").alias("total_ca_ttc"),
    F.sum("total_quantity_sold").alias("total_quantites_vendues"),
    F.sum("number_of_tickets").alias("total_nombre_tickets"),
    F.count("*").alias("nombre_lignes")
)

print("===== KPI SILVER =====")
kpi_silver.show(truncate=False)


# --------------------------------------------------
# KPI calculés à partir de Gold
# --------------------------------------------------

kpi_gold = df_gold.agg(
    F.sum("total_revenue_inc_tax").alias("total_ca_ttc"),
    F.sum("total_quantity_sold").alias("total_quantites_vendues"),
    F.sum("number_of_tickets").alias("total_nombre_tickets"),
    F.count("*").alias("nombre_magasins")
)

print("===== KPI GOLD =====")
kpi_gold.show(truncate=False)

===== KPI SILVER =====


+-------------+-----------------------+--------------------+-------------+
|total_ca_ttc |total_quantites_vendues|total_nombre_tickets|nombre_lignes|
+-------------+-----------------------+--------------------+-------------+
|1202218083.00|33755474               |89100               |21430        |
+-------------+-----------------------+--------------------+-------------+

===== KPI GOLD =====
+-------------+-----------------------+--------------------+---------------+
|total_ca_ttc |total_quantites_vendues|total_nombre_tickets|nombre_magasins|
+-------------+-----------------------+--------------------+---------------+
|1202218083.00|33755474               |89100               |12             |
+-------------+-----------------------+--------------------+---------------+



In [5]:
# --------------------------------------------------
# KPI par magasin
# --------------------------------------------------

kpi_by_store = (
    df_silver
    .groupBy("store_sk")
    .agg(
        F.sum("total_revenue_inc_tax").alias("ca_ttc"),
        F.sum("total_quantity_sold").alias("quantites_vendues"),
        F.sum("number_of_tickets").alias("nombre_tickets")
    )
    .orderBy(F.desc("ca_ttc"))
)

print("===== KPI PAR MAGASIN =====")
kpi_by_store.show(20, truncate=False)

===== KPI PAR MAGASIN =====
+--------+------------+-----------------+--------------+
|store_sk|ca_ttc      |quantites_vendues|nombre_tickets|
+--------+------------+-----------------+--------------+
|7       |103384198.80|2915053          |7645          |
|4       |101130648.13|2832572          |7462          |
|9       |101109360.86|2830684          |7435          |
|0       |101047840.24|2830581          |7456          |
|8       |100916146.68|2821869          |7414          |
|3       |100377573.32|2800824          |7396          |
|1       |100230588.43|2817524          |7422          |
|11      |99910528.38 |2790631          |7449          |
|6       |99040970.11 |2780253          |7284          |
|10      |98490674.24 |2785773          |7409          |
|5       |98329868.79 |2789703          |7369          |
|2       |98249685.02 |2760007          |7359          |
+--------+------------+-----------------+--------------+



In [6]:
# --------------------------------------------------
# Agrégation Silver
# --------------------------------------------------

silver_totals = (
    df_silver
    .agg(
        F.sum("total_revenue_inc_tax").alias("ca_silver"),
        F.sum("total_quantity_sold").alias("quantites_silver"),
        F.sum("number_of_tickets").alias("tickets_silver")
    )
    .first()
)


# --------------------------------------------------
# Agrégation Gold
# --------------------------------------------------

gold_totals = (
    df_gold
    .agg(
        F.sum("total_revenue_inc_tax").alias("ca_gold"),
        F.sum("total_quantity_sold").alias("quantites_gold"),
        F.sum("number_of_tickets").alias("tickets_gold"),
        F.min("global_revenue_inc_tax").alias("ca_global_min"),
        F.max("global_revenue_inc_tax").alias("ca_global_max")
    )
    .first()
)


# --------------------------------------------------
# Comparaison
# --------------------------------------------------

print("===== RÉCONCILIATION SILVER / GOLD =====")

print("CA Silver :", silver_totals["ca_silver"])
print("CA Gold :", gold_totals["ca_gold"])
print("Écart CA :", silver_totals["ca_silver"] - gold_totals["ca_gold"])

print("\nQuantités Silver :", silver_totals["quantites_silver"])
print("Quantités Gold :", gold_totals["quantites_gold"])
print("Écart quantités :",
      silver_totals["quantites_silver"] - gold_totals["quantites_gold"])

print("\nTickets Silver :", silver_totals["tickets_silver"])
print("Tickets Gold :", gold_totals["tickets_gold"])
print("Écart tickets :",
      silver_totals["tickets_silver"] - gold_totals["tickets_gold"])

print("\nCA global Gold min :", gold_totals["ca_global_min"])
print("CA global Gold max :", gold_totals["ca_global_max"])

===== RÉCONCILIATION SILVER / GOLD =====
CA Silver : 1202218083.00
CA Gold : 1202218083.00
Écart CA : 0.00

Quantités Silver : 33755474
Quantités Gold : 33755474
Écart quantités : 0

Tickets Silver : 89100
Tickets Gold : 89100
Écart tickets : 0

CA global Gold min : 1202218083.00
CA global Gold max : 1202218083.00


In [7]:
store_sales_path = (
    DATA_ROOT
    / "silver"
    / "store_sales"
)

df_store_sales = spark.read.parquet(str(store_sales_path))

# --------------------------------------------------
# VALIDATION DU KPI NOMBRE DE TRANSACTIONS
# --------------------------------------------------

print("===== VALIDATION DU NOMBRE DE TICKETS =====")

required_ticket_columns = [
    "ss_store_sk",
    "ss_sold_date_sk",
    "ss_ticket_number"
]

available_columns = df_store_sales.columns

missing_columns = [
    col for col in required_ticket_columns
    if col not in available_columns
]

if missing_columns:
    print("[NON VALIDÉ] Colonnes manquantes :", missing_columns)

else:
    # Contrôle des valeurs nulles
    df_store_sales.select(
        F.count("*").alias("nombre_lignes_source"),
        F.sum(
            F.when(F.col("ss_ticket_number").isNull(), 1)
             .otherwise(0)
        ).alias("tickets_nulls"),
        F.countDistinct("ss_ticket_number").alias(
            "tickets_distincts_identifiant_seul"
        )
    ).show(truncate=False)

    # Décompte distinct des tickets par magasin et date
    tickets_source = (
        df_store_sales
        .groupBy("ss_store_sk", "ss_sold_date_sk")
        .agg(
            F.countDistinct("ss_ticket_number")
            .alias("tickets_source")
        )
        .withColumnRenamed("ss_store_sk", "store_sk")
        .withColumnRenamed("ss_sold_date_sk", "date_sk")
    )

    # Comparaison avec Silver
    comparaison_tickets = (
        df_silver
        .select(
            "store_sk",
            "date_sk",
            F.col("number_of_tickets").alias("tickets_silver")
        )
        .join(
            tickets_source,
            on=["store_sk", "date_sk"],
            how="full_outer"
        )
        .withColumn(
            "ecart_tickets",
            F.col("tickets_silver") - F.col("tickets_source")
        )
    )

    print("===== ÉCARTS ENTRE SOURCE ET SILVER =====")

    comparaison_tickets.select(
        F.count("*").alias("nombre_groupes"),
        F.sum(
            F.when(F.col("tickets_silver").isNull(), 1)
             .otherwise(0)
        ).alias("groupes_absents_silver"),
        F.sum(
            F.when(F.col("tickets_source").isNull(), 1)
             .otherwise(0)
        ).alias("groupes_absents_source"),
        F.sum(
            F.when(F.col("ecart_tickets") != 0, 1)
             .otherwise(0)
        ).alias("groupes_avec_ecart")
    ).show(truncate=False)

    print("Exemples de différences :")

    (
        comparaison_tickets
        .filter(
            F.col("ecart_tickets").isNull()
            | (F.col("ecart_tickets") != 0)
        )
        .show(20, truncate=False)
    )

===== VALIDATION DU NOMBRE DE TICKETS =====


+--------------------+-------------+----------------------------------+
|nombre_lignes_source|tickets_nulls|tickets_distincts_identifiant_seul|
+--------------------+-------------+----------------------------------+
|667579              |0            |89100                             |
+--------------------+-------------+----------------------------------+

===== ÉCARTS ENTRE SOURCE ET SILVER =====


+--------------+----------------------+----------------------+------------------+
|nombre_groupes|groupes_absents_silver|groupes_absents_source|groupes_avec_ecart|
+--------------+----------------------+----------------------+------------------+
|21430         |0                     |0                     |0                 |
+--------------+----------------------+----------------------+------------------+

Exemples de différences :
+--------+-------+--------------+--------------+-------------+
|store_sk|date_sk|tickets_silver|tickets_source|ecart_tickets|
+--------+-------+--------------+--------------+-------------+
+--------+-------+--------------+--------------+-------------+



In [8]:
# Synthèse des KPI validés

kpi_summary = spark.createDataFrame(
    [
        (
            "Chiffre d'affaires TTC",
            str(silver_totals["ca_silver"]),
            "Unité monétaire"
        ),
        (
            "Quantités vendues",
            str(silver_totals["quantites_silver"]),
            "Unités"
        ),
        (
            "Nombre de transactions",
            str(silver_totals["tickets_silver"]),
            "Tickets"
        )
    ],
    ["KPI", "Valeur", "Unite"]
)

kpi_summary.show(truncate=False)

(
    kpi_summary
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(str(Path.home() / "retail-intelligence"
             / "tp4" / "reports" / "kpi_summary"))
)

+----------------------+-------------+---------------+
|KPI                   |Valeur       |Unite          |
+----------------------+-------------+---------------+
|Chiffre d'affaires TTC|1202218083.00|Unité monétaire|
|Quantités vendues     |33755474     |Unités         |
|Nombre de transactions|89100        |Tickets        |
+----------------------+-------------+---------------+



In [9]:
spark.stop()
print("Session Spark fermée.")

Session Spark fermée.
